In [ ]:
from gnews import GNews
import pandas as pd
from datetime import datetime, timedelta
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup


class GoogleNewsCollector:
    def __init__(self, keywords=None, weeks_back=4, results_per_query=100, granularity="daily", delay=0.1, country_lst=[],
                end_date_overwrite = None):
        self.keywords = keywords if keywords else []
        self.weeks_back = weeks_back
        self.max_results = results_per_query
        self.granularity = granularity
        self.delay = delay
        self.country_map = {
            'Australia': 'AU', 'Botswana': 'BW', 'Canada ': 'CA', 'Ethiopia': 'ET', 'Ghana': 'GH', 'India ': 'IN',
            'Indonesia': 'ID', 'Ireland': 'IE', 'Israel ': 'IL', 'Kenya': 'KE', 'Latvia': 'LV', 'Malaysia': 'MY', 'Namibia': 'NA',
            'New Zealand': 'NZ', 'Nigeria': 'NG', 'Pakistan': 'PK', 'Philippines': 'PH', 'Singapore': 'SG', 'South Africa': 'ZA',
            'Tanzania': 'TZ', 'Uganda': 'UG', 'United Kingdom': 'GB', 'United States': 'US', 'Zimbabwe': 'ZW',
            'Czech Republic': 'CZ', 'Germany': 'DE', 'Austria': 'AT', 'Switzerland': 'CH', 'Argentina': 'AR', 'Chile': 'CL',
            'Colombia': 'CO', 'Cuba': 'CU', 'Mexico': 'MX', 'Peru': 'PE', 'Venezuela': 'VE', 'Belgium ': 'BE', 'France': 'FR',
            'Morocco': 'MA', 'Senegal': 'SN', 'Italy': 'IT', 'Lithuania': 'LT', 'Hungary': 'HU', 'Netherlands': 'NL',
            'Norway': 'NO', 'Poland': 'PL', 'Brazil': 'BR', 'Portugal': 'PT', 'Romania': 'RO', 'Slovakia': 'SK', 'Slovenia': 'SI',
            'Sweden': 'SE', 'Vietnam': 'VN', 'Turkey': 'TR', 'Greece': 'GR', 'Bulgaria': 'BG', 'Russia': 'RU', 'Ukraine ': 'UA',
            'Serbia': 'RS', 'United Arab Emirates': 'AE', 'Saudi Arabia': 'SA', 'Lebanon': 'LB', 'Egypt': 'EG',
            'Bangladesh': 'BD', 'Thailand': 'TH', 'China': 'CN', 'Taiwan': 'TW', 'Hong Kong': 'HK', 'Japan': 'JP',
            'Republic of Korea': 'KR'
        }
        self.country_avail_lst = list(set(list(self.country_map.values())) & set(country_lst))
        self.available_countries = list(self.country_map.values()) if len(self.country_avail_lst) == 0 else self.country_avail_lst 
        self.articles = []
        self.visited_urls = set()
        self.end_date_overwrite = datetime.strptime(end_date_overwrite, '%Y-%m-%d') if end_date_overwrite is not None else None #This must be in the string format of 'YYYY-MM-DD'

    def collect(self):
        if not self.keywords:
            raise ValueError("No keywords specified. Please provide a list of search terms.")
        print(f"Starting collection for {len(self.keywords)} keywords across {len(self.available_countries)} countries")
        for country in self.available_countries:
            print(f"\nCountry: {country.upper()}")
            for keyword in self.keywords:
                print(f"Keyword: {keyword}")
                self._collect_for_keyword_country(keyword, country)
                time.sleep(self.delay)

    def _collect_for_keyword_country(self, keyword, country):
        
        gnews = GNews(language='en', country=country)
        gnews.max_results = self.max_results
        gnews.period = None

        date_ranges = self._generate_date_ranges()

        for start_date, end_date in date_ranges:
            gnews.start_date = start_date
            gnews.end_date = end_date

            print(f"  Fetching {start_date} to {end_date}")
            try:
                articles = gnews.get_news(keyword)
                for i, article in enumerate(articles):
                    url = article.get("url")
                    
                    if url in self.visited_urls:
                        print(f"    Skipping (cached): {url}")
                        continue  # Skip already visited URLs
                
                    self.visited_urls.add(url)  # Mark URL as visited
                
                    full_text = ""
                    try:
                        full_text = self._get_article_text_selenium_and_bs(url)
                    except Exception as e:
                        print(f"    Failed to fetch full text for URL: {article.get('url')} | Error: {e}")

                    self.articles.append({
                        "keyword": keyword,
                        "country": country,
                        "title": article.get("title"),
                        "description": article.get("description"),
                        "published_date": article.get("published date"),
                        "url": article.get("url"),
                        "publisher": article.get("publisher", {}).get("title"),
                        "full_text": full_text
                    })
                    print('{} articles retrieved'.format(i+1))
            except Exception as e:
                print(f"  Error fetching: {e}")
            if self.delay > 0:
                time.sleep(self.delay)

            # Triggering autosave
            print('Auto-saving...')
            timestamp = datetime.now().strftime("%Y%m%d_%H%M")
            df = pd.DataFrame(self.articles)
            df.to_parquet('last_autosave_{}.parquet'.format(timestamp), engine='pyarrow', index=False)
            
    def _get_article_text_selenium_and_bs(self, url, timeout=3):
        options = Options()
        options.add_argument("--headless")  # Run in headless mode (no browser UI)
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920x1080")
    
        driver = webdriver.Chrome(options=options)
        try:
            driver.get(url)
            time.sleep(timeout)  # Allow time for the page to load fully
    
            # Get the page source after it has loaded
            page_source = driver.page_source
    
            # Pass the page source to BeautifulSoup for parsing
            soup = BeautifulSoup(page_source, "html.parser")
    
            # Try to find article content using BeautifulSoup
            article_tags = soup.find_all("article")
            if article_tags:
                return "\n".join([el.get_text(strip=True) for el in article_tags if el.get_text(strip=True) != ""])
    
            # Fallback: use all visible paragraphs
            paragraphs = soup.find_all("p")
            text = "\n".join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
            return text
    
        except Exception as e:
            print(f"Error extracting article from {url}: {e}")
            return ""
    
        finally:
            driver.quit()
                
    def _generate_date_ranges(self):
        today = datetime.today().date()

        if self.end_date_overwrite is not None:
            end_date = self.end_date_overwrite
        else:
            end_date = today
        
        days_back = self.weeks_back * 7

        if self.granularity == "daily":
            return [
                (end_date - timedelta(days=i + 1), end_date - timedelta(days=i))
                for i in range(days_back)
            ]
        elif self.granularity == "weekly":
            return [
                (end_date - timedelta(days=7 * (i + 1)), end_date - timedelta(days=7 * i))
                for i in range(self.weeks_back)
            ]
        else:
            raise ValueError("Granularity must be 'daily' or 'weekly'")

    def save_to_parquet(self, filename=None):
        if not filename:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M")
            filename = f"google_news_articles_{self.granularity}_{timestamp}.parquet"
        df = pd.DataFrame(self.articles)
        df.to_parquet(filename, engine='pyarrow', index=False)
        print(f"Saved {len(df)} articles to {filename}")

    def get_dataframe(self):
        return pd.DataFrame(self.articles)

In [ ]:
keywords = [
    "Money Laundering", "Terrorist Financing", "Sanctions Violations",
    "Fraud", "Tax Evasion", "Bribery and Corruption",
    "Insider Trading", "Ponzi Scheme", "Pyramid Scheme", "Trade-Based Money Laundering", 
    "Scandal", "Allegation",
    "Lawsuit", "Money Mule", "Shell Company", "Crime",
    "Scam", "Politician"
]

collector = GoogleNewsCollector(
    keywords=keywords,
    weeks_back=4,
    results_per_query=100,
    granularity="daily",
    delay=1,
    country_lst = ['SG'], # This typically doesn't affect the news retrieved too much, so going with SG
    end_date_overwrite = '2025-04-11'
)

collector.collect()

# Get the raw DataFrame
df = collector.get_dataframe()
collector.save_to_parquet('news_data.parquet')